# P1: Matnga Ishlov Berish Konveyeri
**Mavzu:** Tokenizatsiya, stop-so'zlar, stemming, BoW va TF-IDF
**Kun:** 2-kun amaliyoti — 16-iyun 2026, 09:30–10:50 (80 daqiqa)
**Juftlashgan ma'ruza:** L1 — NLP asoslari va matnni qayta ishlash
**Kapstone modul:** m01 — `TextPreprocessor` (`capstone/modules/m01_text_preprocessor.py`)

**Bugungi maqsadlar:**
1. O'zbek matni uchun regex tokenizer yozish (U+2019 apostrof hisobga olinadi)
2. Stop-so'zlarni filtrlash va sodda stemming pipeline qurish
3. BoW va TF-IDF matritsalar yasash scikit-learn bilan
4. m01 `TextPreprocessor` modulini `capstone/contracts.py` imzolariga muvofiq yozish

| Bo'lim | Vaqt |
|--------|------|
| §1 Muhit tekshiruvi | 3 daq |
| §2 Yaxlit natija (destination first) | 8 daq |
| §3 Tayyor kod bloki — PRIMM | 17 daq |
| §4 Asosiy mavzu — so'nuvchi tayanch | 35 daq |
| §5 Kapstone integratsiya | 13 daq |
| §6 Tadqiqot savoli + yakun | 7 daq |


In [ ]:
# ═══════════════════════════════════════════════════════════════
# §1  Muhit tekshiruvi
# ═══════════════════════════════════════════════════════════════
import random, os, sys
from pathlib import Path
import numpy as np

# Takrorlanish uchun urug' (seed) — asserts stabilligini ta'minlaydi
random.seed(42)
np.random.seed(42)

# Offline rejim (Kaggle sessiyasi va mahalliy ishlatish ikkalasida ishlaydi)
OFFLINE_FALLBACK = True   # <-- bu qiymatni o'zgartirmang

# Checkpoint papkasi tekshiruvi
CHECKPOINT_DIR = Path("d02_checkpoints")
assert CHECKPOINT_DIR.exists(), (
    "d02_checkpoints/ papkasi topilmadi. "
    "Notebookni course/practices/ papkasidan ishga tushiring "
    "yoki: git pull  qilib qayta urinib ko'ring."
)

# CPU-only tekshiruv (1-2-hafta GPU shart emas)
try:
    import torch
    if torch.cuda.is_available():
        print("GPU mavjud — lekin bu amaliyot CPUda ishlatiladi.")
    else:
        print("CPU rejimi: to'g'ri!")
except ImportError:
    print("PyTorch o'rnatilmagan — bu amaliyotda kerak emas.")

# Paket versiyalari
import sklearn, re, collections
print(f"Python    : {sys.version.split()[0]}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"numpy     : {np.__version__}")
print("\n✓ Muhit tayyor.")


---
## §2  Yaxlit Natija — Pirovard Manzil Birinchi! (~8 daqiqa)

Quyida **tugallangan pipeline** bir necha satrda ishga tushadi.
Hozir tushuntirishdan oldin — natijani ko'ring.
Keyingi bo'limlarda shu natijaga qanday erishilishini parchalaymiz.


In [ ]:
# Pirovard natija ko'rinishi (inline versiya — m01 hali yozilmagan)
from pathlib import Path
import re, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# TODO: Stage 0B — kurs Kaggle datasetiga almashtirish
CORPUS_FILE = Path("d02_checkpoints/uz_news_mini.txt")
raw_texts = [l.strip() for l in CORPUS_FILE.open(encoding="utf-8") if l.strip()]

# Mini preprocessing (to'liq versiyasi m01 da)
_APO = re.compile(r"['\u2019\u2018]")
_TOK = re.compile(r"[a-z][a-z']*")
STOPS = {"va", "bu", "bir", "bilan", "da", "ni", "ga", "dan",
         "ham", "uchun", "edi", "ekan", "deb", "lekin", "ammo",
         "yoki", "agar", "chunki", "shu", "esa"}

def _preprocess(text):
    t = _APO.sub("'", text).lower()
    return [w for w in _TOK.findall(t) if len(w) >= 2 and w not in STOPS]

processed = [" ".join(_preprocess(t)) for t in raw_texts]

vec = TfidfVectorizer(max_features=20)
X = vec.fit_transform(processed)

print(f"Korpus hajmi : {len(raw_texts)} hujjat")
print(f"TF-IDF matritsa : {X.shape[0]} × {X.shape[1]}")
print(f"Eng muhim 15 so'z: {list(vec.get_feature_names_out()[:15])}")
print("\n✓ Pipeline ishladi!  Quyida har qadamni o'rganamiz.")

---
## §3  Tayyor Kod Bloki — PRIMM (~17 daqiqa)

### 3A. Matn Yuklash va Encoding

> **Bashorat qiling** — kodni ishlatishdan oldin javob bering:
> Quyidagi kod nimani chop etadi?
> `with open(...)` bloki nima uchun `encoding='utf-8'` kerak?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
# TODO: Stage 0B — quyidagi yo'lni Kaggle Dataset slug-iga almashtirish
CORPUS_FILE = Path("d02_checkpoints/uz_news_mini.txt")

if OFFLINE_FALLBACK or not CORPUS_FILE.exists():
    corpus_path = CORPUS_FILE
else:
    # Onlayn rejim: Kaggle Dataset yoki boshqa manba
    corpus_path = CORPUS_FILE  # hozircha bundled fayl

with open(corpus_path, encoding="utf-8") as f:
    raw_texts = [line.strip() for line in f if line.strip()]

print(f"Yuklandi: {len(raw_texts)} hujjat")
print(f"Birinchi  : '{raw_texts[0]}'")
print(f"Oxirgi    : '{raw_texts[-1]}'")
print(f"O'rtacha uzunlik: {np.mean([len(t.split()) for t in raw_texts]):.1f} so'z/hujjat")


> **Tekshiring:**
> 1. `raw_texts` ro'yxatida nechta element bor?
> 2. `encoding='utf-8'` o'rniga `'latin-1'` yozsangiz nima bo'ladi? Sinab ko'ring.
> 3. `if line.strip()` shart nima uchun kerak?

> **O'zgartiring — shaxsiy korpus!**
> O'zingizdagi biror o'zbek matni faylini `uz_news_mini.txt` o'rniga yuklab ko'ring.
> (Masalan, Wikipedia maqolasi, yangiliklar portali matni, lex.uz hujjati)


### 3B. spaCy: Gaplarni Aniqlash

> **Bashorat qiling:**
> `spacy.blank("xx")` qanday model yuklaydi?
> "Blank" modeli nimasi bilan ajralib turadi?


In [ ]:
# ─── TO'LIQ BERILGAN KOD (PRIMM — periphery) ─────────────────────────────
try:
    import spacy

    # O'zbek tili uchun maxsus spaCy modeli yo'q!
    # 'xx' = til-agnostik bo'sh model (til-atalmas mexanikalar uchun)
    nlp = spacy.blank("xx")
    nlp.add_pipe("sentencizer")      # gap deteksiyasi qoidasi bilan

    sample = raw_texts[0] + "  " + raw_texts[1]
    doc = nlp(sample)

    print("spaCy gaplari:")
    for i, sent in enumerate(doc.sents, 1):
        print(f"  {i}. [{sent.text.strip()[:60]}...]")

    print("\nspaCy tokenlari (birinchi gap):")
    first_sent = next(doc.sents)
    print([t.text for t in first_sent])

except ImportError:
    print("spaCy o'rnatilmagan — mahalliy testda normal.")
    print("Qo'lda tokenizatsiya quyida ko'ramiz.")


> **Tekshiring:**
> 1. spaCy nechta gapni aniqladi? To'g'rimi?
> 2. Apostrofli so'zlar (`o'zbek`, `bo'lib`) tokenizatsiyada to'g'ri ajratilganmi?
> 3. Agar spaCy `o\'zbek` → `["o", "'", "zbek"]` deb bo'lsa — bu nima muammo?

> **O'zbek tili cheklovi (kurs asosi):**
> spaCy ning birorta Uzbek language modeli mavjud emas (2026-yilda).
> Shuning uchun bu kurs **maxsus O'zbek regex-tokenizer** ishlatadi (quyida §4A).
> Bu cheklov kursning asosiy motivatsiyalaridan biri — haftalar davomida yechim quramiz.


In [ ]:
# ─── CHECKPOINT A ─────────────────────────────────────────────────────────
# raw_texts ni keyingi bo'lim uchun saqlab qo'yamiz
# (Agar yuqoridagi katakni o'tkazib yuborganingiz bo'lsa, bu yerdan davom eting)
import json
_ckpt_a = Path("d02_checkpoints/_ckpt_a_raw.json")

if OFFLINE_FALLBACK and _ckpt_a.exists():
    raw_texts = json.loads(_ckpt_a.read_text(encoding="utf-8"))
    print(f"Checkpoint A yuklandi: {len(raw_texts)} hujjat")
else:
    _ckpt_a.write_text(json.dumps(raw_texts, ensure_ascii=False), encoding="utf-8")
    print(f"Checkpoint A saqlandi: {len(raw_texts)} hujjat")


---
## §4  Asosiy Mavzu — So'nuvchi Tayanch (~35 daqiqa)

Bu bo'lim uch bosqichda o'rganamiz:
- **Namuna** (Men ko'rsataman) — to'liq kod + tushuntirish
- **Birgalikda** (Birga to'ldiramiz) — blanklar `# === SIZNING KODINGIZ ===`
- **Mustaqil** (Siz qilasiz) — o'z korpusingiz, scaffold yo'q


### 4A. Namuna: O'zbek uchun Regex Tokenizer

**Muammo:** Standart `split()` va `nltk.word_tokenize` o'zbek apostrofini buzadi.


In [ ]:
import re

# Muammo ko'rsatish: standart split() apostrofni saqlamaydi
text1 = "o'zbek tili bo'yicha o'rganish misollari" # ASCII apostrof 
text2 = "o\u2019zbek tili bo\u2019yicha"  # U+2019 fancy apostrof

print("Standart split():")
print("  ASCII apostrof:", text1.split())
print("  U+2019 apostrof:", text2.split())
print("  Muammo: ayrim tokenayzerlar apostrofda bo'ladi: o'zbek -> [o, zbek]")

# ─── Yechim: Regex tokenizer ────────────────────────────────────────────────
_APOSTROPHE_RE = re.compile(r"['\u2019\u2018]")   # U+2019, U+2018, ASCII '
_TOKEN_RE      = re.compile(r"[a-z][a-z']*")        # harf bilan boshlanadi

def uz_tokenize(text: str) -> list[str]:
    """O'zbek matni uchun regex tokenizer.
    U+2019 (') va ASCII (') apostroflarni standartlashtiradi.
    """
    # 1. Apostrof standartlashtirish
    text = _APOSTROPHE_RE.sub("'", text)
    # 2. Kichik harfga
    text = text.lower()
    # 3. Regex: harf bilan boshlanuvchi tokenlar (apostrof o'z ichida bo'lishi mumkin)
    return _TOKEN_RE.findall(text)

# Sinov
for sample in [text1, text2, "Toshkent O'zbekiston poytaxti"]:
    print(f"  {repr(sample[:40])} -> {uz_tokenize(sample)}")

print("\n✓ Namuna: tokenizer to'g'ri ishlaydi.")


### 4B. Birgalikda: Stop-So'zlarni Filtrlash

Stop-so'zlar (stopwords) — matn ma'nosini ko'tarishga hissa qo'shmaydi.
O'zbek uchun: `va`, `bu`, `bilan`, `da`, `ni`, `ga`, ...


In [ ]:
# O'zbek stop-so'zlar ro'yxati (asosiy)
UZ_STOPWORDS: set[str] = {
    "va", "bu", "bir", "bilan", "da", "ni", "ga", "dan",
    "ham", "uchun", "bo'lgan", "bo'lib", "bo'ldi", "o'z",
    "ular", "u", "men", "biz", "siz", "edi", "ekan", "deb",
    "lekin", "ammo", "yoki", "agar", "chunki", "hali",
    "ko'p", "oz", "shunday", "shu", "esa", "endi",
    "bor", "yo'q", "kerak", "mumkin", "bo'lsa", "bo'lishi",
}

def filter_stopwords(tokens: list[str], stopwords: set[str]) -> list[str]:
    """Stop-so'zlar va qisqa tokenlarni olib tashlaydi.

    Args:
        tokens:    Tokenlar ro'yxati.
        stopwords: Stop-so'zlar to'plami.

    Returns:
        Filtrlangan ro'yxat: stopwords da bo'lmagan va len >= 2 bo'lgan tokenlar.
    """
    # === SIZNING KODINGIZ (taxminan 1-2 qator) ===
    # Maslahat: list comprehension — ikki shart: stopwords va uzunlik
    pass


In [ ]:
# Namunaviy matn uchun tekshiramiz
_sample_raw = uz_tokenize(raw_texts[0])
_sample_filtered = filter_stopwords(_sample_raw, UZ_STOPWORDS)

assert _sample_filtered is not None, (
    "filter_stopwords() None qaytardi. 'pass' o'rniga ro'yxat qaytaring."
)
assert len(_sample_filtered) <= len(_sample_raw), (
    f"Filtrlash xato: natija ({len(_sample_filtered)}) "
    f"asl tokenlardan ({len(_sample_raw)}) ko'p bo'lishi mumkin emas. "
    "filter_stopwords() ni tekshiring."
)
assert all(t not in UZ_STOPWORDS for t in _sample_filtered), (
    "Stop-so'z qoldi! UZ_STOPWORDS ni filtrlash shartiga solishtirayotganingizni tekshiring."
)
assert all(len(t) >= 2 for t in _sample_filtered), (
    "Qisqa token (len<2) qoldi. Uzunlik shartini tekshiring."
)
print(f"✓ To'g'ri! '{raw_texts[0][:50]}...'")
print(f"  {len(_sample_raw)} token → {len(_sample_filtered)} ta saqlandi.")
print(f"  Filtrlangan: {_sample_filtered}")


### 4C. Namuna: Stemming — Ildiz Ajratish

**Maqsad:** `yaxshi`, `yaxshiroq`, `yaxshilik` → hammasi `yaxshi`
Bu lug'at hajmini kamaytiradi va OOV muammosini yumshatadi.


In [ ]:
# ─── SnowballStemmer: boshqa tillar uchun demo ──────────────────────────────
from nltk.stem import SnowballStemmer
import nltk

try:
    english_stemmer = SnowballStemmer("english")
    print("SnowballStemmer (inglizcha demo):")
    for w in ["running", "runner", "runs", "easily", "fairly"]:
        print(f"  {w:12s} → {english_stemmer.stem(w)}")
    print()
except LookupError:
    nltk.download("punkt", quiet=True)

# ─── O'zbek uchun SnowballStemmer yo'q! ─────────────────────────────────────
print("SnowballStemmer o'zbek tilini qo'llab-quvvatlamaydi.")
print("(Qo'llab-quvvatlangan tillar: English, Russian, German, French, ...)")
print()

# ─── O'zbek uchun sodda qo'shimcha qirqish ──────────────────────────────────
_UZ_SUFFIXES: tuple[str, ...] = (
    "larning", "lardan", "larni", "larga", "larim", "laring", "lari",
    "imizdan", "ingizdan",
    "lilik", "ligi", "lik",
    "roqqa", "roq",
    "ishda", "ishi", "ish",
    "ning", "chi",
    "lar", "ni", "da", "dan", "ga",
)

def uz_stem(token: str) -> str:
    """Sodda O'zbek suffix stripping.
    Qo'shimchani olib tashlaydi agar ildiz uzunligi >= 3 bo'lsa.
    """
    for suffix in sorted(_UZ_SUFFIXES, key=len, reverse=True):
        if token.endswith(suffix) and len(token) - len(suffix) >= 3:
            return token[: -len(suffix)]
    return token

# Misol
demo_words = ["yaxshi", "yaxshiroq", "yaxshilik",
              "rivojlanmoqda", "rivojlanish", "rivojlandi",
              "texnologiyalari", "texnologiya"]
print("Sodda O'zbek stemmer (suffix stripping):")
for w in demo_words:
    print(f"  {w:20s} → {uz_stem(w)}")

print("\n Namuna: stemmer ishlaydi. Mukammal emas — bu pedagogik versiya.")


### 4D. Birgalikda: Stemming ni Butun Korpusga Qo'llash


In [ ]:
def preprocess_doc(text: str) -> list[str]:
    """Bitta hujjatni tokenize → filter → stem qiladi.

    Returns:
        Tozalangan stemmed tokenlar ro'yxati.
    """
    # === SIZNING KODINGIZ (3-4 qator) ===
    # 1. uz_tokenize(text) bilan tokenizatsiya
    # 2. filter_stopwords(..., UZ_STOPWORDS) bilan filtrlash
    # 3. Har token uchun uz_stem() qo'llash
    # 4. Ro'yxatni qaytarish
    pass


# Butun korpusni qayta ishlash
processed_corpus: list[list[str]] = []
for text in raw_texts:
    # === SIZNING KODINGIZ (1 qator) ===
    # processed_corpus ga preprocess_doc(text) natijasini qo'shing
    pass


In [ ]:
# ─── Tekshirish ────────────────────────────────────────────────────────────
assert processed_corpus is not None and len(processed_corpus) == len(raw_texts), (
    "processed_corpus ro'yxati to'liq emas. "
    "Barcha hujjatlar uchun preprocess_doc() chaqirayapsizmi?"
)
assert all(isinstance(doc, list) for doc in processed_corpus), (
    "Har element list bo'lishi kerak. preprocess_doc() list qaytaradimi?"
)
assert all(len(doc) > 0 for doc in processed_corpus), (
    "Bo'sh hujjat topildi. preprocess_doc() hech qachon bo'sh list qaytarmasligi kerak."
)

# Lug'at qisqarganini tekshiramiz
raw_vocab  = set(w for t in raw_texts for w in t.lower().split())
stem_vocab = set(w for doc in processed_corpus for w in doc)
print(f"Xom lug'at hajmi : {len(raw_vocab):4d} so'z")
print(f"Tozalangan lug'at: {len(stem_vocab):4d} so'z")
assert len(stem_vocab) < len(raw_vocab), (
    "Lug'at kichraymadi. Filtrlash yoki stemming ishlamoqdami?"
)
print(f"✓ To'g'ri! Lug'at {len(raw_vocab)-len(stem_vocab)} so'zga kichraydi.")
print(f"  Misol: {processed_corpus[0]}")


In [ ]:
# ─── CHECKPOINT B ─────────────────────────────────────────────────────────
_ckpt_b = Path("d02_checkpoints/_ckpt_b_processed.json")

if OFFLINE_FALLBACK and _ckpt_b.exists() and not processed_corpus:
    processed_corpus = json.loads(_ckpt_b.read_text(encoding="utf-8"))
    print(f"Checkpoint B yuklandi: {len(processed_corpus)} hujjat")
else:
    _ckpt_b.write_text(
        json.dumps(processed_corpus, ensure_ascii=False), encoding="utf-8"
    )
    print(f"Checkpoint B saqlandi.")


### 4E. Namuna: TF-IDF Qo'lda Hisob — L1 [I]-Slayd

**Bu katakning natijasi P2 birinchi assert bilan to'g'ridan-to'g'ri bog'liq.**
L1 ma'ruzasida [I3]-slayddagi formula: `TF-IDF(t, d) = TF(t,d) × IDF(t)` bunda
`IDF(t) = ln(N / df(t))` (hech qanday offset va normalizatsiya yo'q).


In [ ]:
# ─── Ma'ruza L1 [I]-slayd bilan bir xil kichik korpus ──────────────────────
docs_l1 = ["nlp qiziq", "python foydali", "nlp foydali"]
N = len(docs_l1)  # 3 hujjat

# So'zlar va ularning hujjat chastotasi (df)
def compute_df(corpus: list[str]) -> dict[str, int]:
    """Har so'z nechta hujjatda uchraydi."""
    df = {}
    for doc in corpus:
        for w in set(doc.split()):
            df[w] = df.get(w, 0) + 1
    return df

df = compute_df(docs_l1)
print("Hujjat chastotalari (df):"); print(df)

# IDF = ln(N / df(t))  -- L1 [I]-slayd formulasi (sklearn dan farqli!)
def idf(word: str, df_dict: dict, n: int) -> float:
    return np.log(n / df_dict[word])

# TF (xom sana) = so'z hujjatda necha marta uchraydi
def tf(word: str, doc: str) -> int:
    return doc.split().count(word)

# D1 = "nlp qiziq" uchun TF-IDF
tfidf_nlp_d1   = tf("nlp",   docs_l1[0]) * idf("nlp",   df, N)
tfidf_qiziq_d1 = tf("qiziq", docs_l1[0]) * idf("qiziq", df, N)

print(f"\nIDF(nlp)   = ln({N}/{df['nlp']}) = {idf('nlp', df, N):.4f}")
print(f"IDF(qiziq) = ln({N}/{df['qiziq']}) = {idf('qiziq', df, N):.4f}")
print(f"TF-IDF(nlp,   D1) = {tf('nlp',docs_l1[0])} × {idf('nlp',df,N):.4f} = {tfidf_nlp_d1:.4f}")
print(f"TF-IDF(qiziq, D1) = {tf('qiziq',docs_l1[0])} × {idf('qiziq',df,N):.4f} = {tfidf_qiziq_d1:.4f}")

# ─── ASSERT: Ma'ruza L1 [I]-slayd bilan solishtiring ─────────────────────
assert abs(tfidf_nlp_d1   - 0.405) < 1e-3, (
    f"TF-IDF(nlp,D1) = {tfidf_nlp_d1:.4f}, kutilgan 0.405. "
    "IDF formulasini tekshiring: ln(N/df), offset yo'q."
)
assert abs(tfidf_qiziq_d1 - 1.099) < 1e-3, (
    f"TF-IDF(qiziq,D1) = {tfidf_qiziq_d1:.4f}, kutilgan 1.099. "
    "IDF formulasini tekshiring: ln(N/df), offset yo'q."
)  # Ma'ruza L1 [I]-slayd bilan solishtiring

print("\n✓ Ikkala assert o'tdi!  Ma'ruza [I]-slayd qiymatlari tasdiqlandi:")
print(f"  TF-IDF(nlp,D1) = {tfidf_nlp_d1:.3f}  [kutilgan: 0.405]")
print(f"  TF-IDF(qiziq,D1) = {tfidf_qiziq_d1:.3f}  [kutilgan: 1.099]")


### 4F. Nega sklearn Standart Natijasi Boshqacha?

> **Muhim o'quv lahzasi** — sklearn `TfidfVectorizer()` defaults boshqacha natija beradi.
> Bu **tuzog'** emas, balki **o'quv darsi**: har formulaning parametrlarini bilish kerak.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

docs_l1 = ["nlp qiziq", "python foydali", "nlp foydali"]

# ─── 1. sklearn DEFAULT ─────────────────────────────────────────────────────
vec_def = TfidfVectorizer()        # smooth_idf=True, norm='l2' (DEFAULT)
X_def   = vec_def.fit_transform(docs_l1).toarray()
names   = list(vec_def.get_feature_names_out())

i_nlp   = names.index("nlp")
i_qiziq = names.index("qiziq")

print("sklearn DEFAULT (smooth_idf=True, norm='l2'):")
print(f"  TF-IDF(nlp,   D1) = {X_def[0, i_nlp]:.4f}   <- ma'ruzadan: 0.405 EMAS!")
print(f"  TF-IDF(qiziq, D1) = {X_def[0, i_qiziq]:.4f}   <- ma'ruzadan: 1.099 EMAS!")

# ─── 2. smooth_idf=False, norm=None ─────────────────────────────────────────
vec_raw = TfidfVectorizer(smooth_idf=False, norm=None)
X_raw   = vec_raw.fit_transform(docs_l1).toarray()
rnames  = list(vec_raw.get_feature_names_out())
ri_nlp  = rnames.index("nlp")
ri_qiziq= rnames.index("qiziq")

print("\nsklearn smooth_idf=False, norm=None:")
print(f"  IDF(nlp)   = {vec_raw.idf_[ri_nlp]:.4f}   <- sklearn: ln(N/df)+1 = 0.405+1")
print(f"  IDF(qiziq) = {vec_raw.idf_[ri_qiziq]:.4f}   <- sklearn: ln(N/df)+1 = 1.099+1")
print(f"  TF-IDF(nlp,   D1) = {X_raw[0, ri_nlp]:.4f}")
print(f"  TF-IDF(qiziq, D1) = {X_raw[0, ri_qiziq]:.4f}")

# ─── Xulosa: 3 ta farq ──────────────────────────────────────────────────────
print(
    "3 ta farq:\n"
    "  1. smooth_idf=True  : IDF = ln((N+1)/(df+1)) + 1  <- nol bo'lim xavfini kamaytiradi\n"
    "  2. +1 offset         : sklearn har doim IDF ga +1 qo'shadi\n"
    "  3. norm='l2'         : har qator L2 normaga bo'linadi, ||qator||=1\n"
    "\n"
    "Ma'ruza [I]-slayd: toza matematika -- IDF = ln(N/df), offset va norm yo'q.\n"
    "sklearn production uchun moslangan.  Ikkalasi ham to'g'ri, maqsad farqli."
)


### 4G. Birgalikda: CountVectorizer — BoW Matritsasi


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Processed corpus ni sklearn uchun string ga aylantirish
corpus_strings = [" ".join(doc) for doc in processed_corpus]

# === SIZNING KODINGIZ (2 qator) ===
# 1. CountVectorizer() yarating
# 2. fit_transform(corpus_strings) chaqirib X_bow oling
vec_bow = None
X_bow   = None


In [ ]:
assert vec_bow is not None and X_bow is not None, (
    "vec_bow yoki X_bow None. "
    "CountVectorizer() yarating va fit_transform() chaqiring."
)
assert X_bow.shape[0] == len(raw_texts), (
    f"Qatorlar soni ({X_bow.shape[0]}) hujjatlar soniga ({len(raw_texts)}) mos emas."
)
assert X_bow.shape[1] > 0, "Lug'at bo'sh. Preprocessing to'g'ri ishlayaptimi?"
assert (X_bow.toarray() >= 0).all(), "BoW da manfiy qiymat bo'lishi mumkin emas."

vocab_sample = list(vec_bow.get_feature_names_out())[:10]
print(f"✓ BoW matritsasi: {X_bow.shape[0]} hujjat × {X_bow.shape[1]} so'z")
print(f"  Lug'at namunasi: {vocab_sample}")
print(f"  Nol bo'lmagan elementlar: {X_bow.nnz} / {X_bow.shape[0]*X_bow.shape[1]}")


### 4H. Birgalikda: TfidfVectorizer


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# === SIZNING KODINGIZ (2 qator) ===
# 1. TfidfVectorizer() yarating
# 2. fit_transform(corpus_strings) chaqirib X_tfidf oling
# Eslatma: bu default parametrlar (smooth_idf=True, norm='l2') bilan ishlaydi
vec_tfidf = None
X_tfidf   = None


In [ ]:
assert vec_tfidf is not None and X_tfidf is not None, (
    "vec_tfidf yoki X_tfidf None. "
    "TfidfVectorizer() yarating va fit_transform() chaqiring."
)
assert X_tfidf.shape == X_bow.shape, (
    f"TF-IDF shakli {X_tfidf.shape} BoW shaklidan {X_bow.shape} farq qilmasligi kerak."
)

# L2 normalizatsiya: har qator norma ≈ 1 bo'lishi kerak
row_norms = np.asarray(X_tfidf.power(2).sum(axis=1)).flatten()
assert np.allclose(row_norms, 1.0, atol=1e-6), (
    "TF-IDF qatorlari L2 normaga kelmadi. "
    "TfidfVectorizer() default norm='l2' ishlatyaptimi?"
)
print(f"✓ TF-IDF matritsasi: {X_tfidf.shape[0]} × {X_tfidf.shape[1]}")
print(f"  Qator norma (birinchi 5): {row_norms[:5].round(4)}")
top_terms = list(vec_tfidf.get_feature_names_out()[
    X_tfidf[0].toarray().argsort()[0][::-1][:5]
])
print(f"  1-hujjat eng muhim so'zlari: {top_terms}")


### 4I. Mustaqil: O'z Korpusingizga Qo'llang

O'zingizning tanlagan matn to'plamingizni yuklang va butun pipeline ni qo'llang.
Scaffold yo'q — yuqorida o'rgangan funksiyalardan foydalaning.


In [ ]:
# === SIZNING KODINGIZ ===
# 1. O'z matn faylini yuklang (yoki quyidagi namunadan foydalaning)
# 2. preprocess_doc() bilan qayta ishlang
# 3. CountVectorizer va TfidfVectorizer ni fit qiling
# 4. Lug'at hajmi va eng muhim 10 so'zni chop eting

# Agar o'z faylingiz bo'lmasa — kichik test matni:
my_texts = [
    "O'zbekistonda sun'iy intellekt rivojlanmoqda.",
    "Yoshlar dasturlash va texnologiyalarni o'rganmoqda.",
    "Milliy texnologiyalar markazi yangi loyihalar yaratmoqda.",
]

# Quyida o'z kodingizni yozing:
my_processed = None
my_vec_tfidf = None
my_X_tfidf   = None


In [ ]:
assert my_processed is not None, (
    "my_processed None. preprocess_doc() bilan korpusni qayta ishlang."
)
assert my_vec_tfidf is not None and my_X_tfidf is not None, (
    "my_vec_tfidf yoki my_X_tfidf None. TfidfVectorizer() qo'llang."
)
assert my_X_tfidf.shape[0] == len(my_texts), (
    "Qatorlar soni mening matnlarim soniga mos emas."
)
assert my_X_tfidf.shape[1] > 0, "TF-IDF lug'at bo'sh."
assert (my_X_tfidf.toarray() >= 0).all(), "TF-IDF da manfiy qiymat."
print(f"✓ Mustaqil: mening TF-IDF matritsam {my_X_tfidf.shape} — to'g'ri!")


---
## §5  Loyihaga Ulash — m01 TextPreprocessor Yozamiz (~13 daqiqa)

Yuqorida qurilgan funksiyalar (`uz_tokenize`, `filter_stopwords`, `uz_stem`,
`preprocess_doc`) ni endi rasmiy kapstone moduliga o'tkazamiz.
Imzolar `capstone/contracts.py` da belgilangan — o'zgartirmaymiz.


In [ ]:
from pathlib import Path

# Repo ildizi (course/practices/ papkasidan 2 daraja yuqori)
REPO_ROOT = Path().resolve().parent.parent
M01_PATH  = REPO_ROOT / "capstone" / "modules" / "m01_text_preprocessor.py"
M01_PATH.parent.mkdir(parents=True, exist_ok=True)

M01_CODE = '''"""
capstone/modules/m01_text_preprocessor.py
TextPreprocessor — O'zbek matni uchun preprocessing pipeline.
Shartnoma: capstone/contracts.py :: TextPreprocessor
P1 (2-kun amaliyoti) da qurilgan; m02-m09, m11, m15 tomonidan ishlatiladi.
"""
from __future__ import annotations

import re
from collections import Counter

_APOSTROPHE_RE = re.compile(r"[\'\\u2019\\u2018]")
_TOKEN_RE      = re.compile(r"[a-z][a-z\']*")

_DEFAULT_STOPWORDS: frozenset[str] = frozenset({
    "va", "bu", "bir", "bilan", "da", "ni", "ga", "dan",
    "ham", "uchun", "bo\'lgan", "bo\'lib", "bo\'ldi", "o\'z",
    "ular", "u", "men", "biz", "siz", "edi", "ekan", "deb",
    "lekin", "ammo", "yoki", "agar", "chunki", "hali",
    "ko\'p", "oz", "shunday", "shu", "esa", "endi",
    "bor", "yo\'q", "kerak", "mumkin", "bo\'lsa", "bo\'lishi",
})

_UZ_SUFFIXES: tuple[str, ...] = (
    "larning", "lardan", "larni", "larga", "larim", "laring", "lari",
    "lilik", "ligi", "lik",
    "roqqa", "roq",
    "ishda", "ishi", "ish",
    "ning", "chi",
    "lar", "ni", "da", "dan", "ga",
)


class TextPreprocessor:
    \'\'\'O\'zbek matni uchun tokenizatsiya + normalizatsiya + stemming pipeline.

    Consumed by: m02, m04, m05, m06, m07, m08, m09, m11, m15.
    \'\'\'

    def __init__(self) -> None:
        self._stopwords: set[str] = set(_DEFAULT_STOPWORDS)

    def _normalize(self, text: str) -> str:
        return _APOSTROPHE_RE.sub("\'", text).lower()

    def _tokenize(self, text: str) -> list[str]:
        return _TOKEN_RE.findall(self._normalize(text))

    def _stem(self, token: str) -> str:
        for suffix in sorted(_UZ_SUFFIXES, key=len, reverse=True):
            if token.endswith(suffix) and len(token) - len(suffix) >= 3:
                return token[: -len(suffix)]
        return token

    def preprocess(self, text: str) -> list[str]:
        if not isinstance(text, str) or not text.strip():
            raise ValueError("preprocess(): text bo\'sh bo\'lmasligi kerak.")
        return [
            self._stem(t)
            for t in self._tokenize(text)
            if len(t) >= 2 and t not in self._stopwords
        ]

    def preprocess_batch(self, texts: list[str]) -> list[list[str]]:
        return [self.preprocess(t) for t in texts]

    def fit_stopwords(self, texts: list[str], max_df: float = 0.85) -> None:
        n_docs = len(texts)
        if n_docs == 0:
            return
        threshold = max(1, int(n_docs * max_df))
        df: Counter[str] = Counter()
        for text in texts:
            df.update(set(self._tokenize(text)))
        for word, count in df.items():
            if count >= threshold:
                self._stopwords.add(word)
'''

M01_PATH.write_text(M01_CODE, encoding="utf-8")
print(f"✓ {M01_PATH} yozildi.")


In [ ]:
import sys

# capstone/modules/ ni Python path ga qo'shish
if str(REPO_ROOT / "capstone" / "modules") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "capstone" / "modules"))

from m01_text_preprocessor import TextPreprocessor

tp = TextPreprocessor()

# Asosiy funksiyalarni tekshiramiz
tokens = tp.preprocess("O'zbek tilida tabiiy tilni qayta ishlash texnologiyalari")
assert isinstance(tokens, list) and len(tokens) > 0, "preprocess() bo'sh list qaytardi."

batch = tp.preprocess_batch(["Salom dunyo", "NLP texnologiyalari"])
assert len(batch) == 2, "preprocess_batch() 2 natija qaytarishi kerak."

try:
    tp.preprocess("")
    assert False, "Bo'sh string uchun ValueError ko'tarilmadi!"
except ValueError:
    pass  # To'g'ri xulq

tp.fit_stopwords(raw_texts, max_df=0.85)
tokens2 = tp.preprocess("O'zbekiston rivojlanmoqda")
# fit_stopwords dan keyin natija o'zgarishi mumkin
assert isinstance(tokens2, list), "fit_stopwords() dan keyin preprocess() ishlamaypti."

print("✓ m01 TextPreprocessor barcha tekshiruvlardan o'tdi!")
print(f"  preprocess() namuna: {tokens}")
print(f"  preprocess_batch() namuna: {batch}")


### 5C. Git — m01 ni Commit Qiling

```bash
git add capstone/modules/m01_text_preprocessor.py capstone/modules/__init__.py
git commit -m "day02: P1 practice + m01 TextPreprocessor"
git push origin HEAD
```


In [ ]:
import subprocess

# Commit (Kaggle da git mavjud bo'lsa)
result = subprocess.run(
    ["git", "add",
     "capstone/modules/m01_text_preprocessor.py",
     "capstone/modules/__init__.py"],
    capture_output=True, text=True, cwd=str(REPO_ROOT)
)
if result.returncode == 0:
    print("✓ git add bajarildi.")
else:
    print(f"git add xato: {result.stderr.strip()}")

# Faqat uncommitted o'zgarishlar bo'lsa commit qiling
status = subprocess.run(
    ["git", "diff", "--cached", "--name-only"],
    capture_output=True, text=True, cwd=str(REPO_ROOT)
)
if status.stdout.strip():
    commit = subprocess.run(
        ["git", "commit", "-m",
         "day02: P1 practice + m01 TextPreprocessor"],
        capture_output=True, text=True, cwd=str(REPO_ROOT)
    )
    print(commit.stdout.strip() or "Commit bajarildi.")
else:
    print("Commit qilish uchun yangi o'zgarish yo'q (allaqachon commit qilingan bo'lishi mumkin).")


---
## §6  Tadqiqot Savoli + Yakun (~7 daqiqa)

**Savol:** Stemming TF-IDF reytingini o'zgartiradimi?
Bir xil ma'noli so'zlar (`yaxshi` / `yaxshiroq` / `yaxshilik`) stemming dan keyin
birlashsa, eng muhim so'zlar reytingi qanday o'zgaradi?


In [ ]:
# Mini tadqiqot: stemming TF-IDF reytingiga ta'siri
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 1. Stemming SIZ (xom tokenlar)
raw_strings = [
    " ".join(uz_tokenize(t)) for t in raw_texts
]

# 2. Stemming BILAN (processed)
processed_strings = [" ".join(doc) for doc in processed_corpus]

vec_nostem  = TfidfVectorizer(max_features=10)
vec_withstem = TfidfVectorizer(max_features=10)

X_nostem   = vec_nostem.fit_transform(raw_strings)
X_withstem = vec_withstem.fit_transform(processed_strings)

print("Stemming SIZ (eng muhim 10 so'z - 1-hujjat):")
nostem_feat = vec_nostem.get_feature_names_out()
nostem_row  = X_nostem[0].toarray().flatten()
for w, s in sorted(zip(nostem_feat, nostem_row), key=lambda x: -x[1])[:5]:
    print(f"  {w:20s}: {s:.4f}")

print("\nStemming BILAN (eng muhim 10 so'z - 1-hujjat):")
wstem_feat = vec_withstem.get_feature_names_out()
wstem_row  = X_withstem[0].toarray().flatten()
for w, s in sorted(zip(wstem_feat, wstem_row), key=lambda x: -x[1])[:5]:
    print(f"  {w:20s}: {s:.4f}")

print("\nXulosa:")
print(f"  Stemming SIZ lug'ati : {len(nostem_feat)} so'z")
print(f"  Stemming BILAN lug'at: {len(wstem_feat)} so'z (max_features=10)")
print("  Reyting o'zgardimi? Qaysi so'zlar birlashdi?")


---
## Yakun

**Bugun nimalar qildik:**
- ✓ O'zbek uchun regex tokenizer (U+2019 apostrof)
- ✓ Stop-so'z filtri + sodda suffix stemmer
- ✓ BoW va TF-IDF matritsalar (CountVectorizer, TfidfVectorizer)
- ✓ TF-IDF formulasini qo'lda va sklearn bilan taqqosladik (0.405/1.099 assert)
- ✓ m01 `TextPreprocessor` ni capstone moduliga yozdik

**Ertaga (P2 — Sentiment klassifikatori):**
m01 → TF-IDF → LogisticRegression / MultinomialNB → m02 SentimentClassifier
Birinchi assert: `abs(ratio - 3.375) < 0.01` (L2 [I2]-slayd)

---

> **Chiqish chipta:** Bugun eng tushunarsiz qolgan narsa nima?
> (Quyidagi katakka yozing — bu savol keyingi darsda muhokama qilinadi)
